In [1]:
# ============================================================================
# 07 — ROBUSTNESS OF THE MAIN RESULT TO HOW THE GROUND TRUTH WAS DEFINED
# ----------------------------------------------------------------------------
# Re-analyses the stored answers on subsets of proteins. No model is queried.
# Needs the OmniPath snapshot in data/raw (not redistributed; see
# data/raw/snapshot_manifest.json). Writes Supplementary Table S6.
# ============================================================================
import hashlib, json
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import binomtest, wilcoxon

ROOT = Path.cwd().parent
PROC, RES = ROOT/"data"/"processed", ROOT/"results"
ONESIDED_CUTOFF = 0.85
A = pd.concat([pd.read_json(p, lines=True) for p in sorted(RES.glob("*.jsonl"))], ignore_index=True)
A = A[A.http==200].copy()
A["abstained"] = A.pred=="not_established"
A["committed"] = A.pred.isin(["stimulates","inhibits"])
A["correct"]   = A.pred.notna() & (A.pred==A.truth)
MAIN = A[(A.arm=="main") & ~A.is_decoy]; DEC = A[(A.arm=="main") & A.is_decoy]
SW, WEB, U = A[A.arm=="swap"], A[A.arm=="web"], A[A.arm=="usually"]
MODELS_ = sorted(MAIN.model.unique()); PROTS = sorted(MAIN.source.unique())
skew = {m: MAIN[(MAIN.model==m) & MAIN.committed].pred.value_counts(normalize=True).max() for m in MODELS_}
ONESIDED = [m for m in MODELS_ if skew[m] > ONESIDED_CUTOFF]
INFORM   = [m for m in MODELS_ if m not in ONESIDED]
def acc(d, c): return d[d.condition==c].correct.mean()
def gap(d, m=None):
    if m is not None: d = d[d.model==m]
    return (acc(d,"habit_agrees") - acc(d,"habit_misleads"))*100
RAW, SUPP = ROOT/"data"/"raw", ROOT/"supplementary"/"tables"
man = json.loads((RAW/"snapshot_manifest.json").read_text())
missing = [f for f in man["files"] if not (RAW/f).exists()]
assert not missing, f"OmniPath snapshot not found in data/raw: {missing}. Results are in Table S6."
bad = [f for f, v in man["files"].items() if hashlib.sha256((RAW/f).read_bytes()).hexdigest() != v["sha256"]]
assert not bad, f"snapshot files differ from the manifest: {bad}"
print("OmniPath snapshot present and matches the manifest checksums")

def is_complex(s):
    return (not isinstance(s,str)) or (not s.strip()) or "COMPLEX" in s.upper() or "_" in s
raw = {n: pd.read_parquet(RAW/f"{n}.parquet") for n in ["causal_ppi","collectri","kinaseextra"]}
parts = []
for n, d in raw.items():
    if n != "collectri": d = d[d.is_directed.astype(bool)]
    st, ih = d.is_stimulation.astype(bool), d.is_inhibition.astype(bool)
    d = d[(st ^ ih).values]
    d = d[~d.source_genesymbol.map(is_complex).values & ~d.target_genesymbol.map(is_complex).values]
    d = d[~d.source.astype(str).str.upper().str.contains("COMPLEX").values
          & ~d.target.astype(str).str.upper().str.contains("COMPLEX").values].copy()
    d["sign"] = d.is_stimulation.astype(bool).map({True:"stimulates", False:"inhibits"})
    d["n_refs"] = d.n_references.fillna(0).astype(int); d["component"] = n
    parts.append(d[["source_genesymbol","target_genesymbol","sign","n_refs","component"]])
E = pd.concat(parts, ignore_index=True)
conf = E.groupby(["source_genesymbol","target_genesymbol"]).sign.nunique()
E["_pk"] = list(zip(E.source_genesymbol, E.target_genesymbol))
E = E[~E._pk.isin(set(conf[conf>1].index))].sort_values("n_refs", ascending=False).drop_duplicates("_pk")

B = pd.read_json(PROC/"sighabit_v2.jsonl", lines=True)
G = pd.read_csv(PROC/"eligible_proteins.csv", index_col=0)
u = B.drop_duplicates("source").set_index("source")
pc = B.pivot_table(index="source", columns="condition", values="component", aggfunc="first")
ALL = set(u.index)

# 1. CollecTRI: signs can default to the TF's general role or to activation
any_ct = set(pc[(pc.habit_agrees=="collectri") | (pc.habit_misleads=="collectri")].index)
agree_ct = set(pc[pc.habit_agrees=="collectri"].index)
Enc = E[(E.n_refs>=2) & (E.component!="collectri")]
unchanged = 0
for s in ALL - any_ct:
    e = Enc[Enc.source_genesymbol==s]; share = (e.sign=="stimulates").mean()
    us = u.loc[s,"protein_usually"]
    unchanged += (us=="stimulates" and share>=.70) or (us=="inhibits" and 1-share>=.62)
print(f"proteins with a CollecTRI question: {len(any_ct)}; of the other {len(ALL-any_ct)}, habit unchanged "
      f"when recomputed without any CollecTRI data: {unchanged}")
# 2. habit flips if single-publication interactions are counted
flip = set(B[np.where(B.src_stim_share_loo>.5,"stimulates","inhibits") != B.protein_usually].source)
# 3. leave-one-out: drop the habit-agreeing interaction and re-test the threshold
loo = set()
for s, r in u.iterrows():
    k = G.loc[s,"stimulates"] if r.protein_usually=="stimulates" else G.loc[s,"inhibits"]
    if (k-1)/(G.loc[s,"total"]-1) < (.70 if r.protein_usually=="stimulates" else .62): loo.add(s)
# 4. support from independent curated resources
res = {}
for d in raw.values():
    for s_, t_, src in zip(d.source_genesymbol, d.target_genesymbol, d.sources):
        res.setdefault((s_,t_), set()).update(x.split("_")[0] for x in str(src).split(";") if x and x != "CollecTRI")
B["n_resources"] = [len(res.get((s_,t_), set())) for s_, t_ in zip(B.source, B.target)]
w = B.pivot_table(index="source", columns="condition", values="n_resources", aggfunc="first")
multi = set(w[(w.habit_agrees>=2) & (w.habit_misleads>=2)].index)

def summarise(label, keep):
    rows = []
    for m in INFORM:
        p = (MAIN[(MAIN.model==m) & MAIN.source.isin(keep)]
             .pivot_table(index="source", columns="condition", values="correct", aggfunc="first").dropna())
        ao = int((p.habit_agrees & ~p.habit_misleads).sum()); bo = int((~p.habit_agrees & p.habit_misleads).sum())
        rows.append(((p.habit_agrees.mean()-p.habit_misleads.mean())*100, binomtest(ao, ao+bo, .5).pvalue if ao+bo else 1))
    g = np.array([r[0] for r in rows]); ps = np.array([r[1] for r in rows])
    return dict(subset=label, proteins=len(keep), median_gap=round(float(np.median(g)),1),
                min_gap=round(float(g.min()),1), max_gap=round(float(g.max()),1),
                models_positive=f"{int((g>0).sum())}/{len(INFORM)}",
                models_McNemar_p_below_005=f"{int((ps<.05).sum())}/{len(INFORM)}", largest_p=float(f"{ps.max():.3g}"))

S6 = pd.DataFrame([
    summarise("All proteins (main analysis)", ALL),
    summarise("No CollecTRI question", ALL - any_ct),
    summarise("Habit-following question not from CollecTRI", ALL - agree_ct),
    summarise("Habit unchanged when single-publication interactions are counted", ALL - flip),
    summarise("Habit threshold met with the tested interaction left out", ALL - loo),
    summarise("Both questions backed by >=2 independent resources", multi),
    summarise("All of the above restrictions combined", ALL - any_ct - flip - loo),
])
S6.to_csv(SUPP/"TableS6_robustness.csv", index=False)
print(S6.to_string(index=False))

# 5. are habit-contradicting partners more obscure?
deg = {}
for d in raw.values():
    for c_ in ["source_genesymbol","target_genesymbol"]:
        for k_, v_ in d[c_].value_counts().items(): deg[k_] = deg.get(k_,0) + v_
B["partner_log_degree"] = np.log1p(B.target.map(lambda t: deg.get(t,0)))
wd = B.pivot_table(index="source", columns="condition", values="partner_log_degree", aggfunc="first")
print(f"\npartner connectivity (log degree), habit-following median {wd.habit_agrees.median():.2f} vs "
      f"habit-contradicting {wd.habit_misleads.median():.2f}; paired Wilcoxon p = {wilcoxon(wd.habit_agrees, wd.habit_misleads).pvalue:.2f}")
print("source database by condition:\n" + pd.crosstab(B.component, B.condition).to_string())
print(f"\nwritten: supplementary/tables/TableS6_robustness.csv")

OmniPath snapshot present and matches the manifest checksums


proteins with a CollecTRI question: 37; of the other 71, habit unchanged when recomputed without any CollecTRI data: 71


                                                          subset  proteins  median_gap  min_gap  max_gap models_positive models_McNemar_p_below_005  largest_p
                                    All proteins (main analysis)       108        30.6     18.5     52.8           14/14                      14/14     0.0375
                                           No CollecTRI question        71        32.4     16.9     56.3           14/14                      13/14     0.0576
                     Habit-following question not from CollecTRI        73        29.5     16.4     53.4           14/14                      13/14     0.0576
Habit unchanged when single-publication interactions are counted        98        30.1     13.3     55.1           14/14                      13/14     0.1650
        Habit threshold met with the tested interaction left out        90        31.1     14.4     54.4           14/14                      13/14     0.1420
              Both questions backed by >=2 ind


partner connectivity (log degree), habit-following median 4.72 vs habit-contradicting 4.66; paired Wilcoxon p = 0.78
source database by condition:
condition    habit_agrees  habit_misleads
component                                
causal_ppi             54              53
collectri              35              37
kinaseextra            19              18

written: supplementary/tables/TableS6_robustness.csv
